# Paper 3 Gate 1 Scale-Up Runner (Vast.ai)

This notebook is for **Vast.ai instances**, not Colab.

It runs the public-benchmark Gate 1 scale-up:

- **MSC valid**: 32 conversations
- **LongMemEval-S cleaned**: 12 conversations
- **model**: `qwen25_15b`
- **budgets**: `0.20,0.35,0.50`

Workflow:

1. clone the repo to the instance,
2. install Python dependencies with `python3`,
3. download and normalize MSC + LongMemEval,
4. run oracle + refinement studies,
5. print the result paths,
6. optionally zip outputs.

Recommended Vast instance:

- `H200` if you want the shortest wall-clock time
- `RTX 4090 / 5090` if you want the best price/performance


In [ ]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
ZIP_URL = "https://github.com/SteveMama/rt-geometry-memory/archive/refs/heads/main.zip"
ZIP_PATH = BASE_DIR / "rt-geometry-memory.zip"
REPO_DIR = BASE_DIR / "rt-geometry-memory"
EXTRACTED_DIR = BASE_DIR / "rt-geometry-memory-main"
PYTHON_BIN = os.environ.get("PYTHON_BIN", "python3")

MODEL = "qwen25_15b"
BUDGETS = "0.20,0.35,0.50"
TARGET_STRIDE = 4
MAX_TARGET_TURNS = 16
LONGMEM_MAX_TURNS = 40
RUN_PREFIX = "paper3_gate1_scaleup"

CLONE_FRESH = True
ZIP_OUTPUTS = True

print({
    "BASE_DIR": str(BASE_DIR),
    "REPO_DIR": str(REPO_DIR),
    "PYTHON_BIN": PYTHON_BIN,
    "MODEL": MODEL,
    "BUDGETS": BUDGETS,
    "TARGET_STRIDE": TARGET_STRIDE,
    "MAX_TARGET_TURNS": MAX_TARGET_TURNS,
    "LONGMEM_MAX_TURNS": LONGMEM_MAX_TURNS,
    "RUN_PREFIX": RUN_PREFIX,
})


In [ ]:
import os
import shlex
import subprocess
import sys

def run(cmd, cwd=None, env=None):
    if isinstance(cmd, str):
        shell = True
        printable = cmd
    else:
        shell = False
        printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"\\n$ {printable}")
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {printable}")

run([PYTHON_BIN, "-V"])
run(["bash", "-lc", "which python3 || true"])
run(["bash", "-lc", "nvidia-smi || true"])


In [ ]:
BASE_DIR.mkdir(parents=True, exist_ok=True)

if CLONE_FRESH and REPO_DIR.exists():
    run(["rm", "-rf", str(REPO_DIR)])
if CLONE_FRESH and EXTRACTED_DIR.exists():
    run(["rm", "-rf", str(EXTRACTED_DIR)])
if CLONE_FRESH and ZIP_PATH.exists():
    run(["rm", "-f", str(ZIP_PATH)])

if not REPO_DIR.exists():
    run(["wget", "-O", str(ZIP_PATH), ZIP_URL], cwd=str(BASE_DIR))
    run(["unzip", "-o", str(ZIP_PATH)], cwd=str(BASE_DIR))
    if REPO_DIR.exists():
        run(["rm", "-rf", str(REPO_DIR)])
    run(["mv", str(EXTRACTED_DIR), str(REPO_DIR)], cwd=str(BASE_DIR))
else:
    run(["bash", "-lc", f"ls -la {shlex.quote(str(REPO_DIR))}"])

print("Repo ready:", REPO_DIR)
run(["bash", "-lc", f"ls -la {shlex.quote(str(REPO_DIR))}"])


In [ ]:
run([PYTHON_BIN, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=str(REPO_DIR))
run([PYTHON_BIN, "-m", "pip", "install", "-e", ".", "huggingface_hub", "tqdm", "pillow"], cwd=str(REPO_DIR))
run([PYTHON_BIN, "-c", "import torch, transformers; print('torch', torch.__version__); print('transformers', transformers.__version__); print('cuda', torch.cuda.is_available()); print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], cwd=str(REPO_DIR))


## Optional Hugging Face login

Uncomment this only if the model download fails.


In [ ]:
# from huggingface_hub import login
# login(token="hf_...")


In [ ]:
MSC_RAW = REPO_DIR / "benchmarks/msc_valid_raw.jsonl"
MSC_NORM = REPO_DIR / "benchmarks/msc_valid_normalized.jsonl"
LONGMEM_RAW = REPO_DIR / "benchmarks/longmemeval_s_cleaned_raw.json"
LONGMEM_NORM = REPO_DIR / "benchmarks/longmemeval_s_cleaned_normalized.jsonl"

run([PYTHON_BIN, "scripts/download_public_benchmark.py", "--benchmark", "msc_valid", "--output", str(MSC_RAW)], cwd=str(REPO_DIR))
run([PYTHON_BIN, "scripts/prepare_public_benchmark_jsonl.py", "--format", "msc", "--input", str(MSC_RAW), "--output", str(MSC_NORM), "--family", "msc_valid"], cwd=str(REPO_DIR))

run([PYTHON_BIN, "scripts/download_public_benchmark.py", "--benchmark", "longmemeval_s_cleaned", "--output", str(LONGMEM_RAW)], cwd=str(REPO_DIR))
run([PYTHON_BIN, "scripts/prepare_public_benchmark_jsonl.py", "--format", "longmemeval", "--input", str(LONGMEM_RAW), "--output", str(LONGMEM_NORM), "--family", "longmemeval_s_cleaned"], cwd=str(REPO_DIR))

for label, path in [("MSC valid", MSC_NORM), ("LongMemEval-S cleaned", LONGMEM_NORM)]:
    line_count = sum(1 for _ in open(path, "r", encoding="utf-8"))
    size_mb = path.stat().st_size / 1e6
    print(f"{label}: {line_count} conversations, {size_mb:.1f} MB -> {path}")


In [ ]:
run([
    "bash",
    "scripts/run_paper3_gate1_scaleup.sh",
    str(MSC_NORM),
    str(LONGMEM_NORM),
    MODEL,
    BUDGETS,
    str(TARGET_STRIDE),
    str(MAX_TARGET_TURNS),
    str(LONGMEM_MAX_TURNS),
    RUN_PREFIX,
], cwd=str(REPO_DIR))


In [ ]:
from pathlib import Path

paths = [
    REPO_DIR / f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_msc_valid_32conv/report.md",
    REPO_DIR / f"results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv/study_report.md",
    REPO_DIR / f"results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv/pairwise_report.md",
    REPO_DIR / f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_longmemeval_s_cleaned_12conv/report.md",
    REPO_DIR / f"results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv/study_report.md",
    REPO_DIR / f"results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv/pairwise_report.md",
]

for path in paths:
    print("\n===", path, "===")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:4000])
    else:
        print("missing")


In [ ]:
if ZIP_OUTPUTS:
    zip_path = REPO_DIR / f"{RUN_PREFIX}_results.zip"
    if zip_path.exists():
        zip_path.unlink()
    run([
        "zip", "-r", str(zip_path),
        f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_msc_valid_32conv",
        f"results/paper3/studies/{RUN_PREFIX}_refinement_msc_valid_32conv",
        f"results/paper3/harm_oracle/{RUN_PREFIX}_oracle_longmemeval_s_cleaned_12conv",
        f"results/paper3/studies/{RUN_PREFIX}_refinement_longmemeval_s_cleaned_12conv",
    ], cwd=str(REPO_DIR))
    print("Wrote:", zip_path)
